In [5]:
import numpy as np, pandas as pd
from pathlib import Path
REP = Path("../reports")

def macro_f1_from_cm(cm):
    tp = np.diag(cm).astype(float)
    p = tp / np.maximum(cm.sum(0), 1); r = tp / np.maximum(cm.sum(1), 1)
    return (2 * p * r / np.maximum(p + r, 1e-12)).mean()

def paired_bootstrap(task, classes, tagA, tagB, B=1000, seed=2026):
    a = pd.read_parquet(REP / f"preds_{tagA}_{task}_test.parquet")
    b = pd.read_parquet(REP / f"preds_{tagB}_{task}_test.parquet")
    m = a.merge(b, on="id", suffixes=("_a", "_b"))
    cid = {c: i for i, c in enumerate(classes)}
    y, pa, pb = (m[c].map(cid).values for c in ("y_true_a", "y_pred_a", "y_pred_b"))
    k, n, rng, d = len(classes), len(y), np.random.default_rng(seed), []
    for _ in range(B):
        i = rng.integers(0, n, n)
        ca = np.bincount(y[i] * k + pa[i], minlength=k * k).reshape(k, k)
        cb = np.bincount(y[i] * k + pb[i], minlength=k * k).reshape(k, k)
        d.append(macro_f1_from_cm(cb) - macro_f1_from_cm(ca))
    lo, hi = np.percentile(d, [2.5, 97.5])
    return round(float(np.mean(d)), 4), round(float(lo), 4), round(float(hi), 4)

SETS = {"recommendation": ["recommended", "not_recommended", "no_idea"],
        "rating_band": ["low", "medium", "high"]}
for task, classes in SETS.items():
    print(task, "semi vs control   (mean diff, 95% CI):", paired_bootstrap(task, classes, "control", "semi"))
    print(task, "control vs tfidf  (mean diff, 95% CI):", paired_bootstrap(task, classes, "tfidf", "control"))

recommendation semi vs control   (mean diff, 95% CI): (0.0002, -0.0007, 0.0011)
recommendation control vs tfidf  (mean diff, 95% CI): (0.0062, 0.0049, 0.0074)
rating_band semi vs control   (mean diff, 95% CI): (-0.0012, -0.0018, -0.0005)
rating_band control vs tfidf  (mean diff, 95% CI): (0.0108, 0.0098, 0.0118)
